# 07. CAM — 모델은 무엇을 보고 판단했는가

**과제 필수 항목.**

정확도가 높다고 끝이 아니다. 모델이 **배경 적혈구나 검은 패딩 모서리**를 보고 맞히고 있다면,
그 정확도는 이 데이터에서만 통하는 숫자다. 판단 근거를 확인해야 한다.

| 절 | 내용 |
|---|---|
| 7-1 | CAM 의 원리와 직접 구현 |
| 7-2 | 클래스별 CAM |
| 7-3 | Grad-CAM 으로 검산 |
| 7-4 | 틀린 사례의 CAM |
| 7-5 | 정량 검증 — 숫자로 확인 (08 가설검정 4의 재료) |

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

import wbc
wbc.use_korean_font()
wbc.NUM_WORKERS = 0

cfg = wbc.load_cfg()
size = int(cfg['image_size'])
model, row = wbc.load_run_model('FINAL')          # 06 에서 학습된 최종 모델
model.eval()
_, ds_test = wbc.make_eval_loader(os.path.join(wbc.DATA_DIR, 'TEST'), image_size=size)
probs, trues = wbc.load_preds('FINAL_TEST', 'test')
preds = probs.argmax(1)
print('모델', cfg['model_name'], '| 테스트', len(ds_test), '장 | 정확도', (preds == trues).mean().round(4))

## 7-1. CAM 의 원리

우리 모델은 마지막이 이렇게 생겼다.

```
... Conv → 특징맵 f (B, C, h, w) → 전역평균풀링(GAP) → Linear(C → 4) → 로짓
```

GAP 는 각 채널의 공간 평균을 낼 뿐이므로, 클래스 c 의 로짓은 이렇게 풀린다.

$$
\text{logit}_c=\sum_k w_{c,k}\cdot\frac{1}{hw}\sum_{h,w} f_k(h,w)
=\frac{1}{hw}\sum_{h,w}\underbrace{\sum_k w_{c,k} f_k(h,w)}_{\displaystyle M_c(h,w)}
$$

즉 **각 위치가 로짓에 얼마나 기여했는지**가 $M_c(h,w)$ 이고, 이것이 CAM 이다.
`GAP → Linear` 구조라서 **근사 없이 정확히** 성립한다.

아래는 그 세 줄짜리 계산을 직접 펼쳐 쓴 것이다.

In [ ]:
@torch.no_grad()
def my_cam(model, x, class_idx=None):
    """x: (3,H,W) 텐서 하나. 반환: (h,w) 히트맵, 예측 클래스, 확률"""
    x = x.unsqueeze(0).to(wbc.device)
    feats  = model.forward_features(x)                 # (1, C, h, w)
    pooled = model.pool(feats).flatten(1)              # (1, C)   GAP
    logits = model.head(pooled)                        # (1, 4)
    pred   = int(logits.argmax(1))
    c      = pred if class_idx is None else class_idx

    w = model.head.weight[c]                           # (C,)  그 클래스의 헤드 가중치
    cam = (feats[0] * w[:, None, None]).sum(0)         # (h, w)  가중합  <- 이 한 줄이 CAM
    cam = F.relu(cam)                                  # 음의 기여는 버린다
    cam = cam / (cam.max() + 1e-8)                     # 0~1 로 정규화
    return cam.cpu().numpy(), pred, torch.softmax(logits, 1)[0].cpu().numpy()

x, y = ds_test[0]
heat, pred, prob = my_cam(model, x)
print('특징맵 해상도에서의 CAM 크기 :', heat.shape, '  (원본', size, 'px 를 이 격자로 요약한 것)')
print(f'정답 {wbc.CLASS_NAMES[y]} / 예측 {wbc.CLASS_NAMES[pred]} (확률 {prob[pred]:.3f})')

In [ ]:
# 히트맵을 원본 크기로 확대해 겹쳐 그린다
img = wbc._to_numpy_img(x)
big = F.interpolate(torch.tensor(heat)[None, None], size=img.shape[:2],
                    mode='bilinear', align_corners=False)[0, 0].numpy()

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].imshow(img); ax[0].set_title('입력'); ax[0].axis('off')
ax[1].imshow(big, cmap='jet'); ax[1].set_title(f'CAM ({heat.shape[0]}x{heat.shape[1]} → 확대)'); ax[1].axis('off')
ax[2].imshow(img); ax[2].imshow(big, cmap='jet', alpha=0.45)
ax[2].set_title('겹쳐 보기'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## 7-2. 클래스별 CAM — 맞힌 사례

네 클래스에서 각각 맞힌 예시를 하나씩 뽑는다.
**빨간 영역이 그 클래스라고 판단한 근거**다.

In [ ]:
correct = np.where(preds == trues)[0]
rng = np.random.RandomState(0)

def pick(c):
    cls_idx = np.where(trues == c)[0]
    ok = np.intersect1d(correct, cls_idx)
    return int(rng.choice(ok if len(ok) else cls_idx))

idx = [pick(c) for c in range(wbc.NUM_CLASSES)]
wbc.show_cam(model, ds_test, idx, method='cam', cols=4,
             title='CAM — 클래스별 맞힌 예시'); plt.show()

In [ ]:
# 같은 이미지를 '다른 클래스라고 본다면 어디를 봤을까' — 클래스별 CAM 비교
i = idx[0]
x, y = ds_test[i]; img = wbc._to_numpy_img(x)
fig, ax = plt.subplots(1, 5, figsize=(16, 3.2))
ax[0].imshow(img); ax[0].set_title(f'정답 {wbc.CLASS_NAMES[y]}'); ax[0].axis('off')
for c in range(4):
    heat, _, prob = my_cam(model, x, class_idx=c)
    big = F.interpolate(torch.tensor(heat)[None, None], size=img.shape[:2],
                        mode='bilinear', align_corners=False)[0, 0].numpy()
    ax[c+1].imshow(img); ax[c+1].imshow(big, cmap='jet', alpha=.45)
    ax[c+1].set_title(f'{wbc.CLASS_NAMES[c][:5]} 확률 {prob[c]:.3f}', fontsize=10); ax[c+1].axis('off')
plt.tight_layout(); plt.show()

클래스마다 히트맵이 달라지는 것이 정상이다. **정답 클래스의 히트맵이 세포에 가장 잘 맞아야** 한다.

## 7-3. Grad-CAM 으로 검산

Grad-CAM 은 특징맵에 대한 **기울기**를 채널 가중치로 쓴다.

$$\alpha_{c,k}=\frac{1}{hw}\sum_{h,w}\frac{\partial\,\text{logit}_c}{\partial f_k(h,w)},\qquad
M_c=\text{ReLU}\Big(\sum_k \alpha_{c,k} f_k\Big)$$

`GAP → Linear` 구조에서는 $\alpha_{c,k}=w_{c,k}/(hw)$ 이므로 **CAM 과 같은 결과**가 나와야 한다.
즉 두 그림이 일치하면 시각화 자체가 신뢰할 만하다는 뜻이고, 다르면 어딘가 잘못된 것이다.

In [ ]:
wbc.show_cam(model, ds_test, idx, method='gradcam', cols=4,
             title='Grad-CAM — 같은 이미지 (CAM 과 일치해야 정상)'); plt.show()

In [ ]:
# 두 방법의 히트맵 상관계수로 정량 확인
cors = []
for i in np.random.RandomState(2).choice(len(ds_test), min(40, len(ds_test)), replace=False):
    x, _ = ds_test[int(i)]
    a, _, _ = my_cam(model, x)
    b, _ = wbc.grad_cam(model, x)
    cors.append(np.corrcoef(a.ravel(), b[0].ravel())[0, 1])
print(f'CAM 과 Grad-CAM 히트맵의 상관계수: 평균 {np.nanmean(cors):.3f} (1에 가까울수록 일치)')

## 7-4. 틀린 사례의 CAM

**여기가 가장 배울 게 많은 부분이다.** 틀린 이유를 눈으로 확인할 수 있다.

전형적인 패턴
- 세포가 프레임 가장자리에 잘려 있다
- 화면에 백혈구가 둘 이상이라 다른 세포를 보고 있다
- 히트맵이 배경으로 흩어져 있다 (근거를 못 찾은 경우)
- 검은 패딩 모서리에 몰려 있다 (**잘못된 단서를 배운 경우 — 가장 위험**)

In [ ]:
wrong = np.where(preds != trues)[0]
if len(wrong):
    wsel = [int(i) for i in np.random.RandomState(0).choice(wrong, min(8, len(wrong)), replace=False)]
    wbc.show_cam(model, ds_test, wsel, method='cam', cols=4, title='틀린 사례 — 모델이 본 곳')
    plt.show()
else:
    print('틀린 사례가 없다 (테스트셋에서 전부 맞힘)')

## 7-5. 정량 검증 — 숫자로 확인

눈으로 "세포를 보는 것 같다"는 **주관적**이다. 측정한다.

**방법**
1. HSV 색공간에서 **진한 보라색(백혈구 핵)** 영역을 잡아 마스크를 만든다
   (01-3 에서 확인: 핵만 진한 보라, 적혈구는 분홍)
2. CAM 히트맵의 전체 질량 중 **마스크 안에 든 비율** `r` 을 잰다
3. 모델이 아무 데나 본다면 `r` ≈ 마스크의 **면적 비율** `a`
4. 세포에 집중한다면 `r` ≫ `a`

이 비교가 **08 의 가설검정 4** 가 된다.

In [ ]:
# 마스크가 실제로 핵을 잡는지 먼저 눈으로 확인
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for k, i in enumerate(idx):
    x, y = ds_test[i]; img = wbc._to_numpy_img(x)
    hsv = wbc._rgb_to_hsv(img)
    mask = ((hsv[..., 0] > 0.60) & (hsv[..., 0] < 0.88) & (hsv[..., 1] > 0.15))
    axes[0, k].imshow(img); axes[0, k].set_title(wbc.CLASS_NAMES[y][:6]); axes[0, k].axis('off')
    axes[1, k].imshow(mask, cmap='gray'); axes[1, k].set_title(f'핵 마스크 ({mask.mean()*100:.1f}%)')
    axes[1, k].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
n_sel = min(200, len(ds_test))
sel = [int(i) for i in np.random.RandomState(1).choice(len(ds_test), n_sel, replace=False)]
ratio, area = wbc.cam_on_cell_ratio(model, ds_test, sel, method='cam')

print(f'표본 {len(ratio)}장')
print(f'  CAM 질량 중 핵 영역 비율  r : 평균 {ratio.mean():.3f}')
print(f'  핵 영역의 면적 비율      a : 평균 {area.mean():.3f}   (아무 데나 볼 때의 기대치)')
print(f'  집중도 배수            r/a : 평균 {(ratio/np.maximum(area,1e-6)).mean():.2f} 배')

plt.figure(figsize=(6.5, 3.6))
plt.hist(area,  bins=30, alpha=.6, label='핵 면적 비율 a (기대치)')
plt.hist(ratio, bins=30, alpha=.6, label='CAM 질량 비율 r (실제)')
plt.xlabel('비율'); plt.ylabel('이미지 수'); plt.legend(); plt.grid(alpha=.3)
plt.title('CAM 이 핵 영역에 얼마나 집중하는가'); plt.show()

pd.DataFrame(dict(ratio=ratio, area=area)).to_csv('cam_ratios.csv', index=False, encoding='utf-8-sig')
print('cam_ratios.csv 저장 — 08 가설검정 4 에서 이 값을 쓴다')

> **한계를 분명히 한다**: 이 마스크는 색으로 만든 근사치이지 의학적 분할이 아니다.
> 그래서 결론은 **"모델이 배경보다 핵 영역을 유의하게 더 본다"** 까지이고,
> "핵의 분엽 수를 세고 있다"까지는 말하지 않는다. **말할 수 있는 것만 말한다.**

## 07 정리

- CAM 의 원리를 직접 구현해 확인했고, Grad-CAM 으로 검산했다
- 맞힌 사례·틀린 사례에서 모델이 본 곳을 확인했다
- "세포를 본다"를 숫자로 재서 `cam_ratios.csv` 에 저장했다

→ 다음: **08_가설검정.ipynb**